##                                             Bronze Transformations

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder.appName("EcomDataPipeline").getOrCreate()


In [0]:
dbutils.fs.ls("abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/to_processed_user_data/")


[FileInfo(path='abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/to_processed_user_data/chunk1.parquet', name='chunk1.parquet', size=731432, modificationTime=1774508727000),
 FileInfo(path='abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/to_processed_user_data/chunk2.parquet', name='chunk2.parquet', size=728056, modificationTime=1774508727000)]

## Discover and process files on arrival using Auto Loader

In [0]:
#Defining Storage RAW PAth
raw_files = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/"
bronze_tables = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/delta/tables/bronze/"


In [0]:
# Configuration
file_location = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/to_processed_user_data/"
checkpoint_location = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/autoloader_checkpoint"
# The destination folder for processed files
move_location = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/users-raw-2/processed_user_data/"
sink_table = bronze_tables + "users" 

# Use Auto Loader to discover and automatically move files
streamingQuery = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", checkpoint_location) \
  .option("cloudFiles.useManagedFileEvents", "true") \
  .option("cloudFiles.cleanSource", "MOVE") \
  .option("cloudFiles.cleanSource.moveDestination", move_location) \
  .option("cloudFiles.cleanSource.retentionDuration", "1 second") \
  .load(file_location) \
  .writeStream \
  .format("delta") \
  .option("mergeSchema", "true") \
  .option("checkpointLocation", checkpoint_location) \
  .trigger(availableNow = True) \
  .start(sink_table)

# ... your existing code ...
query = streamingQuery # Ensure you assign the .start() to a variable

# 1. Run the stream as you have it
query = streamingQuery
query.awaitTermination()

# 2. Manual Move Backup (If Auto Loader fails)
files = dbutils.fs.ls(file_location)
for f in files:
    if "autoloader_checkpoint" not in f.path and f.name.endswith(".parquet"):
        try:
            print(f"Moving {f.name}...")
            dbutils.fs.mv(f.path, move_location + f.name)
        except Exception:
            # File was likely already moved by Auto Loader's cleanSource
            print(f"File {f.name} already moved by background process.")


Moving chunk1.parquet...
Moving chunk2.parquet...


In [0]:
display(spark.sql(f"SELECT * FROM cloud_files_state('{checkpoint_location}')"))


path,size,create_time,discovery_time,commit_time,archive_time,source_id,processed_time,ingestion_state,archive_mode,move_location
landing-zone-2@stecomdatapoccin/users-raw-2/to_processed_user_data/chunk2.parquet,728056,2026-03-26T07:17:45Z,2026-03-26T07:42:42.667Z,null,null,0,2026-03-26T07:42:45.386Z,INGESTED,null,null
landing-zone-2@stecomdatapoccin/users-raw-2/to_processed_user_data/chunk1.parquet,731432,2026-03-26T07:17:45Z,2026-03-26T07:42:42.665Z,null,null,0,2026-03-26T07:42:45.386Z,INGESTED,null,null
landing-zone-2@stecomdatapoccin/users-raw-2/to_processed_user_data/chunk3.parquet,733179,2026-03-26T07:17:45Z,2026-03-26T07:42:42.667Z,null,null,0,2026-03-26T07:42:45.386Z,INGESTED,null,null


In [0]:
""" Read parquet file from /mnt/ecomdata1/user-raw-2 folder
userDF = spark.read.format("parquet")\
    .option("header",'true')\
    .option("inferSchema",'true')\
    .load(raw_files +"users-raw-2/to_processed_user_data")

userDF.write.format("delta")\
    .mode("overwrite")\
    .save( bronze_tables + "users")"""

' Read parquet file from /mnt/ecomdata1/user-raw-2 folder\nuserDF = spark.read.format("parquet")    .option("header",\'true\')    .option("inferSchema",\'true\')    .load(raw_files +"users-raw-2/to_processed_user_data")\n\nuserDF.write.format("delta")    .mode("overwrite")    .save( bronze_tables + "users")'

##                                             Silver Transformations 


In [0]:
silver_tables = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/delta/tables/silver/"


In [0]:
#Loading Table from bronze_tables Path
usersDF = spark.read.format("delta").load( bronze_tables + "users")


In [0]:
usersDF = usersDF.withColumn("countrycode", upper(col("countrycode")))

# Handling multiple languages elegantly with `expr` and `case when`
usersDF = usersDF.withColumn("language_full", 
                             expr("CASE WHEN language = 'EN' THEN 'English' " +
                                  "WHEN language = 'FR' THEN 'French' " +
                                  "ELSE 'Other' END"))

# Correcting potential data entry errors in `gender` column
usersDF = usersDF.withColumn("gender", 
                             when(col("gender").startswith("M"), "Male")
                             .when(col("gender").startswith("F"), "Female")
                             .otherwise("Other"))

# Using `regexp_replace` to clean `civilitytitle` values
usersDF = usersDF.withColumn("civilitytitle_clean", 
                             regexp_replace("civilitytitle", "(Mme|Ms|Mrs)", "Ms"))

# Derive new column `years_since_last_login` from `dayssincelastlogin`
usersDF = usersDF.withColumn("years_since_last_login", col("dayssincelastlogin") / 365)

# Calculate age of account in years and categorize into `account_age_group`
usersDF = usersDF.withColumn("account_age_years", round(col("seniority") / 365, 2))
usersDF = usersDF.withColumn("account_age_group",
                             when(col("account_age_years") < 1, "New")
                             .when((col("account_age_years") >= 1) & (col("account_age_years") < 3), "Intermediate")
                             .otherwise("Experienced"))

# Add a column with the current year for comparison
usersDF = usersDF.withColumn("current_year", year(current_date()))

# Creatively combining strings to form a unique user descriptor
usersDF = usersDF.withColumn("user_descriptor", 
                             concat(col("gender"), lit("_"), 
                                    col("countrycode"), lit("_"), 
                                    expr("substring(civilitytitle_clean, 1, 3)"), lit("_"), 
                                    col("language_full")))

usersDF = usersDF.withColumn("flag_long_title", length(col("civilitytitle")) > 10)

usersDF = usersDF.withColumn("hasanyapp", col("hasanyapp").cast("boolean"))
usersDF = usersDF.withColumn("hasandroidapp", col("hasandroidapp").cast("boolean"))
usersDF = usersDF.withColumn("hasiosapp", col("hasiosapp").cast("boolean"))
usersDF = usersDF.withColumn("hasprofilepicture", col("hasprofilepicture").cast("boolean"))


usersDF = usersDF.withColumn("socialnbfollowers", col("socialnbfollowers").cast(IntegerType()))
usersDF = usersDF.withColumn("socialnbfollows", col("socialnbfollows").cast(IntegerType()))

usersDF = usersDF.withColumn("productspassrate", col("productspassrate").cast(DecimalType(10, 2)))
usersDF = usersDF.withColumn("seniorityasmonths", col("seniorityasmonths").cast(DecimalType(10, 2)))
usersDF = usersDF.withColumn("seniorityasyears", col("seniorityasyears").cast(DecimalType(10, 2)))

usersDF = usersDF.withColumn("dayssincelastlogin",
                             when(col("dayssincelastlogin").isNotNull(),
                                  col("dayssincelastlogin").cast(IntegerType()))
                             .otherwise(0))  #

In [0]:
#Writing Table from silver_tables Path
usersDF.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(silver_tables + "users")

##                                             Gold Transformations 

In [0]:
#Defining Tables Path for silver & gold
gold_tables = "abfss://landing-zone-2@stecomdatapoccin.dfs.core.windows.net/delta/tables/gold/"

In [0]:
# Read the necessary Silver tables
silver_users = spark.read.format("delta").load( silver_tables + "users")

In [0]:
silver_users.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(gold_tables + "users")